# Lab 1 — Distance Metrics

**Day 04 · Distance-Based ML & MLOps · Cisco AI/ML Training**

---

## Goals

1. Represent two loans as **numeric vectors** in feature space.
2. Compute **Euclidean**, **Manhattan**, and **cosine** distance/similarity.
3. Explain why **scaling** matters before distance-based algorithms (preview Lab 2 KNN).
4. Compare how each metric reacts when you change the loan pair.

> **Quick check:** cosine similarity ≈ **0.90** · Euclidean ≈ **69370** · Manhattan ≈ **71367**

**Data:** `data/lending-club/lending_club_sample.csv` (**1000** rows)

## Why this matters

KNN (Lab 2) classifies a loan by finding its **nearest neighbors** in feature space. The distance formula you choose — and whether features are **scaled** — changes who counts as "close." Day 2 NumPy norms and dot products reappear here on real Lending Club vectors.

## Three ways to measure "closeness"

| Metric | Formula (vectors a, b) | Intuition |
|--------|------------------------|----------|
| **Euclidean** | $\sqrt{\sum (a_i - b_i)^2}$ | Straight-line distance |
| **Manhattan** | $\sum \|a_i - b_i\|$ | Grid / city-block distance |
| **Cosine similarity** | $\frac{a \cdot b}{\|a\|\|b\|}$ | Angle between directions (scale-free) |

Cosine **distance** = $1 - \text{similarity}$. Values near **1** similarity mean the vectors point in similar directions even if magnitudes differ.

## Linear algebra refresher

Each loan is a **vector** in ℝ⁵. Distance metrics use **norms** and **dot products**:

| Operation | NumPy | Role in this lab |
|-----------|-------|------------------|
| L2 norm | `np.linalg.norm(v)` | Euclidean distance |
| L1 norm | `np.abs(v).sum()` | Manhattan distance |
| Dot product | `np.dot(a, b)` | Cosine similarity numerator |

## Distance metric cheat sheet

| Task | Code |
|------|------|
| Euclidean | `np.sqrt(np.sum((a - b) ** 2))` |
| Manhattan | `np.sum(np.abs(a - b))` |
| Cosine sim | `np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))` |
| Difference vector | `a - b` |

---

## 0. Locate the Lending Club file

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

GH_ROOT = Path.cwd().resolve()
if GH_ROOT.name == "notebooks":
    GH_ROOT = GH_ROOT.parents[2]
elif (GH_ROOT.parent / "notebooks").is_dir() and (GH_ROOT.parents[1] / "requirements-student.txt").is_file():
    GH_ROOT = GH_ROOT.parents[1]
else:
    for parent in [GH_ROOT, *GH_ROOT.parents]:
        if (parent / "requirements-student.txt").is_file():
            GH_ROOT = parent
            break

LENDING_CLUB_CSV = GH_ROOT / "data" / "lending-club" / "lending_club_sample.csv"
DEFAULT_STATUSES = {"Charged Off", "Late (31-120 days)"}
NUMERIC_FEATURES = ["loan_amnt", "int_rate", "annual_inc", "dti", "installment"]

print("GH root:", GH_ROOT)
print("Lending Club CSV exists:", LENDING_CLUB_CSV.is_file())
print("Path:", LENDING_CLUB_CSV.relative_to(GH_ROOT))


### 0b. Load **1000** loans and derive `default` label

In [ ]:
df = pd.read_csv(LENDING_CLUB_CSV)
df["default"] = df["loan_status"].isin(DEFAULT_STATUSES).astype(int)

print(f"rows: {len(df)}")
print(f"columns: {len(df.columns)}")
print(f"default rate: {df['default'].mean():.4f}")
display(df[NUMERIC_FEATURES + ["default"]].head(3))


### 0c. Feature ranges — why scale matters

In [ ]:
ranges = df[NUMERIC_FEATURES].agg(["min", "max", "mean"]).round(2)
display(ranges)
print("annual_inc spans orders of magnitude vs int_rate — distance without scaling is dominated by income.")


### 0d. `describe()` on numeric features

In [ ]:
display(df[NUMERIC_FEATURES].describe().round(2))


---

## 1. Pick two loan feature vectors (rows 0 and 1)

In [ ]:
sample = df[NUMERIC_FEATURES].iloc[:2].to_numpy(dtype=float)
a, b = sample[0], sample[1]

print(f"point A (first loan):  {a.round(2)}")
print(f"point B (second loan): {b.round(2)}")
print(f"vector shapes: a={a.shape}, b={b.shape}")


### 1b. Difference vector `a - b`

In [ ]:
diff = a - b
print("difference vector:", diff.round(2))
print("L2 norm of diff (Euclidean):", round(float(np.linalg.norm(diff)), 4))
print("L1 norm of diff (Manhattan):", round(float(np.abs(diff).sum()), 4))


### 1c. Per-feature absolute gaps

In [ ]:
gaps = pd.DataFrame({
    "feature": NUMERIC_FEATURES,
    "loan_a": a.round(2),
    "loan_b": b.round(2),
    "abs_diff": np.abs(a - b).round(2),
}).sort_values("abs_diff", ascending=False)
display(gaps)


---

## 2. Compute distances (checkpoint pair)

In [ ]:
euclidean = float(np.sqrt(np.sum((a - b) ** 2)))
manhattan = float(np.sum(np.abs(a - b)))
cosine_similarity = float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))
cosine_distance = 1.0 - cosine_similarity

print(f"Euclidean distance: {euclidean:.4f}")
print(f"Manhattan distance: {manhattan:.4f}")
print(f"cosine similarity: {cosine_similarity:.4f}")
print(f"cosine distance: {cosine_distance:.4f}")


### Reading the numbers

Euclidean and Manhattan distances are **large** because `annual_inc` and `installment` are on very different scales than `int_rate` and `dti`. Cosine similarity is high (~**0.90**) because the vectors share similar **direction** — a hint that cosine can be useful when magnitude matters less than pattern.

### 2b. Dot product breakdown for cosine

In [ ]:
dot_ab = float(np.dot(a, b))
norm_a = float(np.linalg.norm(a))
norm_b = float(np.linalg.norm(b))
print(f"a·b = {dot_ab:.2f}")
print(f"||a|| = {norm_a:.2f}, ||b|| = {norm_b:.2f}")
print(f"cosine = dot / (||a|| * ||b||) = {dot_ab / (norm_a * norm_b):.4f}")


### 2c. Manhattan vs Euclidean ratio

In [ ]:
ratio = manhattan / euclidean
print(f"Manhattan / Euclidean ratio: {ratio:.4f}")
print("In high dimensions Manhattan often grows faster relative to Euclidean (curse of dimensionality preview).")


---

## 3. Per-feature contribution to Manhattan distance

In [ ]:
contrib = pd.DataFrame(
    {
        "feature": NUMERIC_FEATURES,
        "abs_diff": np.abs(a - b).round(2),
        "pct_of_manhattan": (np.abs(a - b) / manhattan * 100).round(1),
    }
).sort_values("abs_diff", ascending=False)
display(contrib)


### 3b. Which feature dominates?

In [ ]:
top_feature = contrib.iloc[0]["feature"]
print(f"Largest Manhattan contributor: {top_feature}")
print(f"Share of total Manhattan distance: {contrib.iloc[0]['pct_of_manhattan']:.1f}%")


---

## 4. Extension — try rows 10 and 50 (`iloc[9]`, `iloc[49]`)

In [ ]:
a2 = df[NUMERIC_FEATURES].iloc[9].to_numpy(dtype=float)
b2 = df[NUMERIC_FEATURES].iloc[49].to_numpy(dtype=float)

euclidean2 = float(np.sqrt(np.sum((a2 - b2) ** 2)))
manhattan2 = float(np.sum(np.abs(a2 - b2)))
cosine_sim2 = float(np.dot(a2, b2) / (np.linalg.norm(a2) * np.linalg.norm(b2)))

print(f"pair (rows 10 & 50) — Euclidean: {euclidean2:.2f}, cosine: {cosine_sim2:.4f}")


In [ ]:
compare = pd.DataFrame(
    {
        "pair": ["rows 0 & 1", "rows 10 & 50"],
        "euclidean": [euclidean, euclidean2],
        "manhattan": [manhattan, manhattan2],
        "cosine_sim": [cosine_similarity, cosine_sim2],
    }
)
display(compare.round(4))


### Which metric changed most?

Compare the **relative** change in each column between the two pairs. Cosine similarity often shifts less than raw Euclidean when income scales differ.

In [ ]:
pct_change = compare.set_index("pair").pct_change().iloc[1] * 100
print("Percent change (rows 10&50 vs rows 0&1):")
for metric, val in pct_change.items():
    print(f"  {metric}: {val:+.1f}%")


---

## 5. Why scaling matters (preview for Lab 2)

Without scaling, KNN treats a $10,000 difference in `annual_inc` as far more important than a 5-point `int_rate` gap. Lab 2 wraps `StandardScaler` in the pipeline for this reason.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaled = scaler.fit_transform(df[NUMERIC_FEATURES].iloc[:2])
sa, sb = scaled[0], scaled[1]

euclidean_scaled = float(np.sqrt(np.sum((sa - sb) ** 2)))
cosine_scaled = float(np.dot(sa, sb) / (np.linalg.norm(sa) * np.linalg.norm(sb)))

print(f"Euclidean (scaled): {euclidean_scaled:.4f}")
print(f"cosine similarity (scaled): {cosine_scaled:.4f}")


### 5b. Unscaled vs scaled distance comparison

In [ ]:
scale_compare = pd.DataFrame({
    "metric": ["Euclidean", "cosine_sim"],
    "unscaled": [euclidean, cosine_similarity],
    "scaled_pair": [euclidean_scaled, cosine_scaled],
}).round(4)
display(scale_compare)


### 5c. Z-score one feature manually (`int_rate`)

In [ ]:
ir = df["int_rate"].to_numpy(dtype=float)
z0 = (ir[0] - ir.mean()) / ir.std()
z1 = (ir[1] - ir.mean()) / ir.std()
print(f"int_rate z-scores for loans 0 and 1: {z0:.3f}, {z1:.3f}")
print(f"abs z-gap: {abs(z0 - z1):.3f}")


---

## 6. Geometry — unit vectors and angle

In [ ]:
ua, ub = a / np.linalg.norm(a), b / np.linalg.norm(b)
angle_rad = np.arccos(np.clip(np.dot(ua, ub), -1.0, 1.0))
angle_deg = np.degrees(angle_rad)
print(f"angle between loans 0 and 1: {angle_deg:.1f} degrees")
print(f"cosine similarity matches cos(angle): {cosine_similarity:.4f}")


### 6b. Unit vectors preserve direction only

In [ ]:
print("unit vector a:", np.round(ua, 4))
print("unit vector b:", np.round(ub, 4))
print("||ua||:", round(float(np.linalg.norm(ua)), 6))


---

## 7. Mini distance matrix (first 5 loans)

In [ ]:
block = df[NUMERIC_FEATURES].iloc[:5].to_numpy(dtype=float)
n = len(block)
dist_matrix = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        dist_matrix[i, j] = np.linalg.norm(block[i] - block[j])

labels = [f"loan_{i}" for i in range(n)]
display(pd.DataFrame(dist_matrix.round(1), index=labels, columns=labels))


### 7b. Nearest neighbor to loan 0 (Euclidean, unscaled)

In [ ]:
dists_from_0 = dist_matrix[0, 1:]
nearest_idx = int(np.argmin(dists_from_0)) + 1
print(f"Nearest to loan 0 (among loans 1-4): loan_{nearest_idx}, distance={dists_from_0.min():.1f}")


---

## 8. When to use which metric

| Scenario | Prefer |
|----------|--------|
| Features on similar scales | Euclidean |
| High-dimensional sparse text | Cosine |
| Grid-like movement / robust to outliers | Manhattan |
| Mixed-scale tabular (this lab) | **Scale first**, then Euclidean (Lab 2) |

---

## 9. Try it yourself

1. Compute cosine similarity between loans at `iloc[0]` and `iloc[99]`.
2. Which pair is more similar — (0,1) or (0,99) by cosine?
3. Does Euclidean agree?

In [ ]:
# Your code here (optional — solution in next cell)


In [ ]:
c = df[NUMERIC_FEATURES].iloc[99].to_numpy(dtype=float)
cos_0_99 = float(np.dot(a, c) / (np.linalg.norm(a) * np.linalg.norm(c)))
euc_0_99 = float(np.linalg.norm(a - c))
print(f"cosine(0,99): {cos_0_99:.4f} vs cosine(0,1): {cosine_similarity:.4f}")
print(f"Euclidean(0,99): {euc_0_99:.2f} vs Euclidean(0,1): {euclidean:.2f}")


### 9b. Pairwise cosine similarities (first 4 loans)

In [ ]:
block4 = df[NUMERIC_FEATURES].iloc[:4].to_numpy(dtype=float)
cos_pairs = []
for i in range(4):
    for j in range(i + 1, 4):
        vi, vj = block4[i], block4[j]
        sim = float(np.dot(vi, vj) / (np.linalg.norm(vi) * np.linalg.norm(vj)))
        cos_pairs.append({"loan_i": i, "loan_j": j, "cosine_sim": round(sim, 4)})
display(pd.DataFrame(cos_pairs))


### 9c. Feature correlation heatmap preview

In [ ]:
corr = df[NUMERIC_FEATURES].corr().round(3)
display(corr)


### 9d. StandardScaler on full column (preview Lab 2)

In [ ]:
from sklearn.preprocessing import StandardScaler
full_scaled = StandardScaler().fit_transform(df[NUMERIC_FEATURES])
print("scaled matrix shape:", full_scaled.shape)
print("column means after scale:", np.round(full_scaled.mean(axis=0), 4))


---

## 10. Checkpoint summary

In [ ]:
assert abs(cosine_similarity - 0.8960) < 0.01
assert abs(euclidean - 69370.1405) < 100
assert abs(manhattan - 71367.0300) < 100
print("Numbers match — you're good.")



---

## Reflection

1. Which feature contributed most to Manhattan distance for the first pair?
2. When would cosine similarity be preferred over Euclidean distance?
3. How does scaling change the story before applying KNN?
